# 🏔️ Soil Image Classification Model Training
**AI Smart Agriculture Assistant**

This notebook trains a CNN model for classifying soil types from images.

> ⚠️ **Enable GPU:** Runtime → Change runtime type → T4 GPU

## Step 1: Setup

In [ ]:
!pip install -q tensorflow matplotlib numpy

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## Step 2: Download Soil Dataset (Directly from GitHub)

This bypasses all Kaggle restrictions (403 errors) by downloading a verified 7-class Soil Image Dataset directly from a public open-source repository.

In [ ]:
# Download Soil Classification dataset directly
print('Downloading dataset... (this may take a minute)')
!wget -q -O soil_data.zip https://github.com/Phantom-fs/Soil-Classification-Dataset/archive/refs/heads/main.zip
print('Unzipping dataset...')
!unzip -q soil_data.zip -d /content/

# Clean up paths
!mv /content/Soil-Classification-Dataset-main/Orignal-Dataset /content/soil_dataset
!rm -rf /content/Soil-Classification-Dataset-main soil_data.zip

# Find dataset directory
import glob
for root, dirs, _ in os.walk('/content/soil_dataset'):
    if len(dirs) >= 4:  # Looking for class folders
        DATASET_PATH = root
        break
else:
    DATASET_PATH = '/content/soil_dataset'

print(f'\nDataset path: {DATASET_PATH}')
classes = sorted(os.listdir(DATASET_PATH))
classes = [c for c in classes if os.path.isdir(os.path.join(DATASET_PATH, c))]
print(f'Soil types found: {len(classes)}')
print(f'Classes: {classes}')

# Count images per class
for cls in classes:
    count = len(os.listdir(os.path.join(DATASET_PATH, cls)))
    print(f'  {cls}: {count} images')

## Step 3: Prepare Data

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.3,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True
)

val_gen = val_datagen.flow_from_directory(
    DATASET_PATH, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation', shuffle=False
)

NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = list(train_gen.class_indices.keys())
print(f'\nClasses: {NUM_CLASSES} — {CLASS_NAMES}')
print(f'Train: {train_gen.samples}, Val: {val_gen.samples}')

## Step 4: Build & Train Model

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7)
]

# Phase 1: Feature Extraction
print('=== Phase 1: Feature Extraction ===')
history1 = model.fit(train_gen, validation_data=val_gen, epochs=15, callbacks=callbacks)

# Phase 2: Fine-tuning
print('\n=== Phase 2: Fine-Tuning ===')
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss='categorical_crossentropy', metrics=['accuracy'])
history2 = model.fit(train_gen, validation_data=val_gen, epochs=10, callbacks=callbacks)

## Step 5: Evaluate

In [ ]:
loss, accuracy = model.evaluate(val_gen)
print(f'\n✅ Validation Accuracy: {accuracy*100:.2f}%')

acc = history1.history['accuracy'] + history2.history['accuracy']
val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(acc, label='Train'); ax1.plot(val_acc, label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(True)
loss_h = history1.history['loss'] + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
ax2.plot(loss_h, label='Train'); ax2.plot(val_loss, label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

## Step 6: Convert to TFLite & Download

In [ ]:
model.save('soil_image_model.h5')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('soil_image_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f'✅ TFLite model saved!')
print(f'Size: {len(tflite_model)/(1024*1024):.1f} MB')

with open('soil_labels.txt', 'w') as f:
    for name in CLASS_NAMES:
        f.write(name + '\n')
print(f'Labels: {CLASS_NAMES}')

In [ ]:
from google.colab import files
files.download('soil_image_model.tflite')
files.download('soil_labels.txt')
files.download('soil_image_model.h5')

print('\n🎉 Done! Copy soil_image_model.tflite to:')
print('   app/src/main/assets/soil_image_model.tflite')